In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import emcee
import sys
import fitsio
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import FastICA

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *

%load_ext autoreload
%autoreload 2

In [ ]:
#table = create_merged_file(BGS_Y3_ANY_FULL_FILE, IAN_BGS_Y3_MERGED_FILE_LOA, "3")
table = Table.read(IAN_BGS_Y3_MERGED_FILE_LOA)

In [ ]:
# Use a previous run of the group catalog, which has nice cuts applied to it, to get a clean sample of galaxies to do PCA on.
y3_fullfoot_cat = deserialize(cat.bgs_y3_pzp_2_6_c3_1p)
df = y3_fullfoot_cat.all_data
df = df.loc[df['Z_ASSIGNED_FLAG'] == AssignedRedshiftFlag.DESI_SPEC.value] # Only DESI observed ones of course
df = df.loc[~df['IS_SAT']] # Centrals only! 
df = df.loc[:, ['TARGETID', 'VMAX']] # Drop everything else from group catalog

print(len(df))

In [ ]:
names = [name for name in table.colnames if len(table[name].shape) <= 1] 
dft = table[names].to_pandas()

df = df.merge(dft, on='TARGETID', how='inner')
print(len(df))

In [ ]:
# TODO I don't knwo why these cuts from preprocess did not work, so replicate for now
maxlum = np.log10(4e11)
df = df[df['LOG_L_GAL'] < maxlum]
print(len(df))

In [ ]:
# Derive some columns to include in PCA / ICA
df['G-R'] = df['ABS_MAG_G_K_BEST'] - df['ABS_MAG_R_K_BEST']
#df['G-Z'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_Z']
#df['R-Z'] = df['ABSMAG01_SDSS_R'] - df['ABSMAG01_SDSS_Z']
df['SSFR'] = df['SFR'] / np.power(10, df['LOGMSTAR'])
#df['LOGSSFR'] = np.log10(df['SSFR'])
df['LOGSSFR'] = soft_clip_floor(np.log10(df['SSFR']), floor=-14)

# Set inf to na
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
# TODO galaxy density in X mpc around each target (Andres)
# VDISP is 250 km/s, the default value far too often to be used as a feature. 
# ZZSUN (metallicity) is pure junk unfortunately.
# AGE is Light Weighted Age. It has two weird peaks at 6 and 12 Gyr. Shares info with Dn4000. We will not use this.
# TODO HALPHA_EW, HBETA_EW could try
# TODO Consider how to incorporate redshift 'Z'. Could do PCA on residuals after redshift trend removal?
#feature_cols = ['ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_Z', 'LOGMSTAR', 'DN4000_MODEL', 'SHAPE_R_KPC', 'SERSIC', 'LOGSSFR', 'AGE']
feature_cols = ['ABS_MAG_R_K_BEST', 'G-R']

In [ ]:
np.isnan(df[feature_cols]).any() # Check for nans in features, which PCA can't handle. Should be False. 

## Examine properties

In [ ]:
print(df.columns)

In [ ]:
# Examine age in z bins
zbins = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(zbins)-1):
    zmin, zmax = zbins[i], zbins[i+1]
    print(f"Redshift bin: {zmin} - {zmax}")
    subset = df.loc[(df['Z'] >= zmin) & (df['Z'] < zmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['AGE'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    axes[1].hist(b['AGE'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('AGE')
axes[0].set_yscale('log')
axes[1].set_xlabel('AGE')
axes[1].set_yscale('log')
axes[1].legend()

In [ ]:
# Examine VISP in z bins
zbins = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(zbins)-1):
    zmin, zmax = zbins[i], zbins[i+1]
    print(f"Redshift bin: {zmin} - {zmax}")
    subset = df.loc[(df['Z'] >= zmin) & (df['Z'] < zmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['VDISP'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    axes[1].hist(b['VDISP'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)


    # 250 is a special value in the FSF code. What percent of them are at 250?
    print(f"Percent of quiescent galaxies with VDISP=250 in this redshift bin: {(r['VDISP'] == 250).mean():.2%}")
    print(f"Percent of star-forming galaxies with VDISP=250 in this redshift bin: {(b['VDISP'] == 250).mean():.2%}")
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('VDISP')
axes[0].set_yscale('log')
axes[1].set_xlabel('VDISP')
axes[1].set_yscale('log')
axes[1].legend()


In [ ]:
# Examine VDISP in magnitude bins
magbins = np.arange(-24, -16, 1.0)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(magbins)-1):
    magmin, magmax = magbins[i], magbins[i+1]
    print(f"Mag bin: {magmin} - {magmax}")
    subset = df.loc[(df['ABSMAG01_SDSS_R'] >= magmin) & (df['ABSMAG01_SDSS_R'] < magmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['VDISP'], bins=50, label=f'{magmin} to {magmax}', histtype='step', lw=1.5)
    axes[1].hist(b['VDISP'], bins=50, label=f'{magmin} to {magmax}', histtype='step', lw=1.5)


    # 250 is a special value in the FSF code. What percent of them are at 250?
    print(f"Q : VDISP=250: {(r['VDISP'] == 250).mean():.2%}")
    print(f"SF: VDISP=250: {(b['VDISP'] == 250).mean():.2%}")
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('VDISP')
axes[0].set_yscale('log')
axes[1].set_xlabel('VDISP')
axes[1].set_yscale('log')
axes[1].legend()


In [ ]:
# Plot LOGSSFR for QUIESCENT vs STAR-FORMING to see if it looks reasonable
bins = np.linspace(-14, -8, 100)
plt.hist(df['LOGSSFR'], bins=bins, alpha=0.5, label='All')
plt.legend()
plt.yscale('log')
plt.title('LOGSSFR for Quiescent vs Star-forming')


In [ ]:
# Plot LOGSSFR for QUIESCENT vs STAR-FORMING to see if it looks reasonable
red = df.loc[df['QUIESCENT']]
blue = df.loc[~df['QUIESCENT']] 
bins = np.linspace(-14, -8, 100)
plt.hist(red['LOGSSFR'], bins=bins, alpha=0.5, label='Quiescent')
plt.hist(blue['LOGSSFR'], bins=bins, alpha=0.5, label='Star-forming')
plt.legend()
plt.yscale('log')
plt.title('LOGSSFR for Quiescent vs Star-forming')

In [ ]:
# Let's examine P(g-r | Mr)
magbins = np.array([-22.6473, -21.9413, -21.488,  -21.091,  -20.7092, -20.3206, -19.9002, -19.4147, -18.8037, -17.9408, -16.4536])
mid = ((magbins[:-1] + magbins[1:]) / 2).round(1)
df['magbin'] = pd.cut(df['ABS_MAG_R_K_BEST'], bins=magbins, labels=mid)

# groupby magbin and show g-r distribution for each bin
clean = df #df.loc[df['Z'] > 0.005] cut not needed if not usign vmax
plt.figure(figsize=(10, 6))
bins = np.linspace(-0.25, 1.5, 50)
for name, group in clean.groupby('magbin'):
    plt.hist(group['G-R'], bins=bins, label=f'Mr={name}', histtype='step', lw=1.5, density=True)#, weights=1/group['VMAX']) # Not really needed since mag bins pretty narrow
    plt.yscale('log')
#plt.legend()
plt.title('P(g-r | Mr) in DR2 Centrals')

In [ ]:
# Compute and show the correlation matrix of the features
all_feature_cols = ['ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_Z', 
                'LOGMSTAR', 'DN4000_MODEL', 'SHAPE_R_KPC', 'SERSIC', 'LOGSSFR',
                'AGE']
features = df[all_feature_cols].dropna()
all_feature_cols[0] = '$M_r$'
all_feature_cols[1] = '$M_g$'
all_feature_cols[2] = '$M_z$'
all_feature_cols[3] = '$\log M_*$'
all_feature_cols[4] = '$Dn4000$'
all_feature_cols[5] = '$R_e$ (kpc)'
all_feature_cols[6] = '$n_s$'
all_feature_cols[7] = '$\log SSFR$'
all_feature_cols[8] = '$LW Age$'
corr_matrix = np.corrcoef(features.T)  # or features.corr() as a DataFrame
plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, interpolation='none')
plt.colorbar()
plt.xticks(ticks=np.arange(len(all_feature_cols)), labels=all_feature_cols, rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(all_feature_cols)), labels=all_feature_cols)

for i in range(len(all_feature_cols)):
    for j in range(len(all_feature_cols)):
        plt.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.title('Galaxy Correlation Matrix')
plt.tight_layout()
plt.show()

## PCA/ICA

In [ ]:
# Seperate into test/training sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1, random_state=894)
n_features = len(feature_cols)

# Drop rows with any NaN in the selected features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)
all_clean = df.dropna(subset=feature_cols)
print(f"Training samples after NaN drop: {len(train_clean)} / {len(train)}")
print(f"Test samples after NaN drop: {len(test_clean)} / {len(test)}")

# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
test_scaled = scaler.transform(test_clean[feature_cols])
all_scaled = scaler.transform(all_clean[feature_cols])

# Compare the original components to the scaled ones
for f in feature_cols:
    plt.hist(train_clean[f], bins=100, alpha=0.5, label=f'Original')
    plt.hist(train_scaled[:, feature_cols.index(f)], bins=100, alpha=0.5, label=f'Scaled')
    plt.legend()
    plt.title(f'{f}')
    plt.yscale('log')
    plt.show()

 # Fit full PCA on training set
pca_full = PCA(n_components=n_features)
pca_full.fit(train_scaled)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, n_features+1), pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('PCA Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance per Component')

axes[1].plot(range(1, n_features+1), np.cumsum(pca_full.explained_variance_ratio_), 'ko-')
axes[1].axhline(0.95, color='red', linestyle='--', label='95%')
axes[1].axhline(0.99, color='orange', linestyle='--', label='99%')
axes[1].set_xlabel('Number of PCA Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit ICA on training set
ica = FastICA(n_components=n_features, whiten='unit-variance', random_state=597)
train_ica = ica.fit_transform(train_scaled)

# Reorder: sort ICs by abs correlation with LOGMHALO (descending)
# Fix sign: IC most correlated with LOGMHALO should have positive correlation
corr = np.corrcoef(train_ica.T, train_clean['ABS_MAG_R_K_BEST'].values)[:-1, -1]
order = np.argsort(-np.abs(corr))
signs = -np.sign(corr[order])
# Apply permutation + sign flip to ica.components_ and ica.mixing_
ica.components_ = (ica.components_[order].T * signs).T
ica.mixing_ = (ica.mixing_.T[order].T) * signs  # update mixing accordingly
del train_ica # Done with this now that we've fixed the order and signs and have the model.


In [ ]:
# Print feature loadings
print("\nPCA Feature loadings:")
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(pca_full.components_.shape[0])]
)
print(loadings.round(3).to_string())

# Print feature loadings
print("\nICA Feature loadings:")
loadings = pd.DataFrame(
    ica.components_.T,
    index=feature_cols,
    columns=[f'IC{i+1}' for i in range(ica.components_.shape[0])]
)
print(loadings.round(3).to_string())

In [ ]:
# Apply PCA transformation to training and test sets
all_pca_full = pca_full.transform(all_scaled)
all_with_pca = pd.concat([df.reset_index(drop=True), pd.DataFrame(all_pca_full, columns=[f'PCA{i+1}' for i in range(all_pca_full.shape[1])])], axis=1)

train_ica_full = ica.transform(train_scaled)
test_ica_full = ica.transform(test_scaled)
all_ica_full = ica.transform(all_scaled)
train_with_ica = pd.concat([train_clean.reset_index(drop=True), pd.DataFrame(train_ica_full, columns=[f'ICA{i+1}' for i in range(train_ica_full.shape[1])])], axis=1)
test_with_ica = pd.concat([test_clean.reset_index(drop=True), pd.DataFrame(test_ica_full, columns=[f'ICA{i+1}' for i in range(test_ica_full.shape[1])])], axis=1)   
all_with_ica = pd.concat([all_clean.reset_index(drop=True), pd.DataFrame(all_ica_full, columns=[f'ICA{i+1}' for i in range(all_ica_full.shape[1])])], axis=1)
complete_sample = pd.concat([all_with_ica.reset_index(drop=True), pd.DataFrame(all_pca_full, columns=[f'PCA{i+1}' for i in range(all_pca_full.shape[1])])], axis=1) 

# Assert no nans in the final output
assert np.isnan(all_pca_full).sum() == 0
assert np.isnan(all_ica_full).sum() == 0


In [ ]:
complete_sample = pd.concat([all_with_ica.reset_index(drop=True), pd.DataFrame(all_pca_full, columns=[f'PCA{i+1}' for i in range(all_pca_full.shape[1])])], axis=1) 

In [ ]:
# Examine some scatter plots of the a sample of objects in the original feature space, ICA space, and PCA space
sample = complete_sample.sample(1000, random_state=687)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[2].scatter(sample['ABS_MAG_R_K_BEST'], sample['G-R'], alpha=0.5)
axes[2].set_xlabel('ABS_MAG_R_K_BEST')
axes[2].set_ylabel('G-R')
axes[2].set_title('Original Feature Space')
axes[0].scatter(sample['ICA1'], sample['ICA2'], alpha=0.5)
axes[0].set_xlabel('ICA1')
axes[0].set_ylabel('ICA2')
axes[0].set_title('ICA Space')
axes[1].scatter(sample['PCA1'], sample['PCA2'], alpha=0.5)
axes[1].set_xlabel('PCA1')
axes[1].set_ylabel('PCA2')
axes[1].set_title('PCA Space')
plt.tight_layout()
plt.show()

In [ ]:
col = 'ICA2'
red = all_with_ica.loc[all_with_ica['QUIESCENT'] == 1]
blue = all_with_ica.loc[all_with_ica['QUIESCENT'] == 0]
bins = np.linspace(-5, 5, 100)
plt.hist(all_with_ica[col], bins=bins, alpha=1.0, label='All', color='k', histtype='step', linewidth=1.5)
plt.hist(red[col], bins=bins, alpha=0.5, label='Quiescent', color='red', histtype='step', linewidth=1.5)
plt.hist(blue[col], bins=bins, alpha=0.5, label='Star-forming', color='blue', histtype='step', linewidth=1.5)
plt.legend()
plt.title(f'{col} Distribution')

# Now do this in fixed magnitude bins to see if the separation is just due to magnitude or if it holds at fixed magnitude.
magnitude_bins = np.linspace(-24, -17, 8)  # Example magnitude bins
fig, axes = plt.subplots(1, 1, figsize=(8, 6))
for i in range(len(magnitude_bins)-1):
    magmin, magmax = magnitude_bins[i], magnitude_bins[i+1]
    print(f"Mag bin: {magmin} - {magmax}")
    subset = all_with_ica.loc[(all_with_ica['ABS_MAG_R_K_BEST'] >= magmin) & (all_with_ica['ABS_MAG_R_K_BEST'] < magmax)]
    plt.hist(subset[col], bins=bins, alpha=0.6, label=f'{magmin} to {magmax}', histtype='step', linewidth=1.5, density=True)
plt.legend()
plt.title(f'{col} Distribution in Magnitude Bins')

In [ ]:
# Show histograms of the first few ICA components to see if they look reasonable
for i in range(min(5, n_features)):
    plt.hist(train_ica_full[:, i], bins=100, alpha=0.5, label=f'Train ICA {i+1}', density=True)
    plt.hist(test_ica_full[:, i], bins=100, alpha=0.5, label=f'Test ICA {i+1}', density=True)
    plt.legend()
    plt.title(f'ICA Component {i+1} Distribution')
    plt.show()

In [ ]:
# First let's see if correlation matrix is now diagonal for both
corr_ica = pd.DataFrame(all_ica_full).corr()
corr_pca = pd.DataFrame(all_pca_full).corr()
corr_orig = pd.DataFrame(all_clean[feature_cols]).corr()

import seaborn as sns
# Show
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
sns.heatmap(corr_ica, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'ICA{i+1}' for i in range(corr_ica.shape[0])], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'ICA{i+1}' for i in range(corr_ica.shape[0])], rotation=0)    
plt.title('Correlation Matrix of ICA Components')
plt.subplot(1, 3, 2)
sns.heatmap(corr_pca, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'PCA{i+1}' for i in range(corr_pca.shape[0])], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'PCA{i+1}' for i in range(corr_pca.shape[0])], rotation=0)
plt.title('Correlation Matrix of PCA Components')
plt.subplot(1, 3, 3)
sns.heatmap(corr_orig, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
if len(feature_cols) == 4:
    plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=45, ha='right')
    plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=0)
plt.title('Correlation Matrix of Original Properties')
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise MI for each method
def pairwise_mi(X, random_state=31579):
    n = X.shape[1]
    mi = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mi[i, j] = mutual_info_regression(
                    X[:, j:j+1], X[:, i], random_state=random_state, n_jobs=-1
                )[0]
    return mi

from sklearn.feature_selection import mutual_info_regression

idx = np.random.choice(range(all_ica_full.shape[0]), 10000, replace=False)
sample_ica = all_ica_full[idx]
idx = np.random.choice(range(all_pca_full.shape[0]), 10000, replace=False)
sample_pca = all_pca_full[idx]
idx = np.random.choice(range(all_clean.shape[0]), 10000, replace=False)
sample_orig = all_clean.iloc[idx]
sample_orig = sample_orig[feature_cols].to_numpy()

mi_ica = pairwise_mi(sample_ica)
mi_pca = pairwise_mi(sample_pca)
mi_orig = pairwise_mi(sample_orig)

# Too slow! TODO check this somehow though...
#mi_ica = pairwise_mi(all_ica_full)
#mi_pca = pairwise_mi(all_pca_full)
#mi_orig = pairwise_mi(all_clean[feature_cols].to_numpy())

vmax = max(mi_ica.max(), mi_pca.max(), mi_orig.max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels_ica = [f'IC{i+1}' for i in range(n_features)]
labels_pca = [f'PC{i+1}' for i in range(n_features)]
labels_orig = [f'{col}' for col in feature_cols]
# Replace ABS_MAG_R_K_BEST with $M_r$ in labels_orig for better display
labels_orig = [r'$M_r$' if col == 'ABS_MAG_R_K_BEST' else col for col in labels_orig]
for ax, mi, labels, title in zip(axes, [mi_ica, mi_pca, mi_orig], [labels_ica, labels_pca, labels_orig], ['ICA', 'PCA', 'Original']):
    im = ax.imshow(mi, cmap='Oranges', vmin=0, vmax=vmax)
    ax.set_xticks(range(n_features)); ax.set_xticklabels(labels)
    ax.set_yticks(range(n_features)); ax.set_yticklabels(labels)
    for r in range(n_features):
        for c in range(n_features):
            ax.text(c, r, f'{mi[r,c]:.3f}', ha='center', va='center', fontsize=16)
    ax.set_title(f'{title} MI')

fig.colorbar(im, ax=axes[2], location='right')
plt.tight_layout()

print(f"ICA mean off-diagonal MI: {mi_ica[mi_ica > 0].mean():.4f}")
print(f"PCA mean off-diagonal MI: {mi_pca[mi_pca > 0].mean():.4f}")
print(f"Original mean off-diagonal MI: {mi_orig[mi_orig > 0].mean():.4f}")

## Autoencoder reconstruction as a function of components kept

In [ ]:
n_components_range = range(1, n_features + 1)
mse_train = []
mse_test = []

for n in n_components_range:
    pca_n = PCA(n_components=n)
    # Encode (project to n components) then decode (reconstruct)
    train_encoded = pca_n.fit_transform(train_scaled)
    train_reconstructed = pca_n.inverse_transform(train_encoded)
    test_encoded = pca_n.transform(test_scaled)
    test_reconstructed = pca_n.inverse_transform(test_encoded)

    mse_train.append(mean_squared_error(train_scaled, train_reconstructed))
    mse_test.append(mean_squared_error(test_scaled, test_reconstructed))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_components_range, mse_train, 'ko-', label='Train')
ax.plot(n_components_range, mse_test, 'rs-', label='Test')
ax.set_xlabel('Number of PCA Components Kept')
ax.set_ylabel('Reconstruction MSE (standardized units)')
ax.set_title('PCA Autoencoder Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'N Components':<15} {'Cumul. Var':<15} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 60)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
for i, n in enumerate(n_components_range):
    print(f"{n:<15} {cumvar[i]:<15.4f} {mse_train[i]:<15.4f} {mse_test[i]:<15.4f}")

# Save Model

In [ ]:
# Save off the model and scaler for later use
if len(feature_cols) == 4: 
    pkl_file = GAL_PCA_MODEL_FILE
    txt_file = GAL_PCA_MODEL_TEXT_FILE
if len(feature_cols) == 2:
    pkl_file = GAL_2P_MODEL_FILE
    txt_file = GAL_2P_MODEL_TEXT_FILE

model_to_save = ica

joblib.dump((model_to_save, scaler, feature_cols), pkl_file)

# Export PCA model to plain text for C++ consumption
# Format: header lines with # comments, then data blocks
n_comp, n_feat = model_to_save.components_.shape

with open(txt_file, 'w') as f:
    f.write(f"# Galaxy model for C++ use\n")
    f.write(f"# features: {' '.join(feature_cols)}\n")
    f.write(f"# n_features: {n_feat}\n")
    f.write(f"# n_components: {n_comp}\n")
    f.write(f"\n# scaler_mean ({n_feat})\n")
    np.savetxt(f, scaler.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# scaler_scale ({n_feat})\n")
    np.savetxt(f, scaler.scale_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# latent_mean_in_scaled_space ({n_feat})\n")
    np.savetxt(f, model_to_save.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# latent_components ({n_comp} x {n_feat})  — row i is eigenvector i\n")
    np.savetxt(f, model_to_save.components_, fmt='%.10e')
    if (model_to_save is ica):
        f.write(f"\n# latent_mixing_matrix ({n_feat} x {n_comp})  — for inverse transform\n")
        np.savetxt(f, model_to_save.mixing_, fmt='%.10e')  

print(f"Wrote C++ model text file to {txt_file}")

In [ ]:
all_with_ica

In [ ]:
clean = all_with_ica.loc[all_with_ica['Z'] > 0.02]

from footprintmanager import FootprintManager
mgr = FootprintManager()
frac_area = mgr.get_footprint('Y3') / DEGREES_ON_SPHERE
fluxlimits = np.zeros(len(clean['Z'])) + 19.5  
fluxlimits = np.where(clean['PHOTSYS'] == b"N", fluxlimits + 0.04, fluxlimits)

# Will impos a zmin of 0.02 for this version
clean['VMAX'] = get_max_observable_volume(clean['ABS_MAG_R'], 0.02, 0.5, fluxlimits, frac_area)

# Let's cut a few objects that were messing this up
#clean = all_with_ica.loc[all_with_ica['VMAX'] > 500]

In [ ]:
# VMAX version of density functions

# Normal property density functions
fig, axes = plt.subplots(1, len(feature_cols), figsize=(5*len(feature_cols), 5))
for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.set_xlabel(f'{col}')
    ax.set_yscale('log')
    ax.set_ylabel('Density [1/(Mpc/h)^3]')
    ax.grid(True, ls="--", lw=0.5, alpha=0.5)

    for col in feature_cols:
        data = clean[col].values

        b = make_adaptive_density_bins(data, 300, 0, 0.4, 5e-6)
        print(f"{col}: data [{data.min():.2f}, {data.max():.2f}]  bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

        d = np.diff(b)
        # Count up by vmax in each bin
        counts, _ = np.histogram(data, bins=b, weights=1/clean['VMAX'])
        centers = 0.5 * (b[:-1] + b[1:])
        density = counts / d

        # Now to be able to handle very low densities on the edges, add 5 points that are 0.1 * the edge density and 0.1 away.
        # We just need the abundance matching to handle the first few rare objects without pulling values that are so extremal as to be not real.
        scale_length = 0.01 * (np.max(b) - np.min(b))  # 1% of the width of the features space we used
        tail =  density[0] * 10.0**np.arange(-1, -6, -1)
        centers_tail = centers[0] - scale_length * np.arange(1, 6)
        tail2 = density[-1] * 10.0**np.arange(-1, -6, -1)
        centers_tail2 = centers[-1] + scale_length * np.arange(1, 6)

        density = np.concatenate([tail[::-1], density, tail2])
        centers = np.concatenate([centers_tail[::-1], centers, centers_tail2])

        # For magnitudes, reverse the innate order because we want more negative to be on the right (later in the file)
        if 'MAG' in col.upper():
            centers = centers[::-1]
            density = density[::-1]

        output_data = np.column_stack((centers, density))

        ax = axes[feature_cols.index(col)]
        ax.plot(centers, density, drawstyle='steps-mid', label=col, color=pp.get_color(feature_cols.index(col)))
        np.savetxt(get_gal_prop_density_func_file(col), output_data)

plt.tight_layout()
plt.show()


In [ ]:
# ICA Density Functions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax2 = axes[1]
ax.set_xlabel('ICAx Value')
ax2.set_xlabel('ICAx Value (zoom)')
ax.set_yscale('log')
ax2.set_yscale('log')
ax.set_ylabel('Density [1/(Mpc/h)^3]')
ax2.set_ylabel('Density [1/(Mpc/h)^3]')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)
ax2.grid(True, ls="--", lw=0.5, alpha=0.5)

for i in range(all_ica_full.shape[1]):
    colname = f'ICA{i+1}'
    data = clean[colname].values

    b = make_adaptive_density_bins(data, 300, 0, 0.25, 1e-6)
    print(f"{colname}: data [{data.min():.2f}, {data.max():.2f}]  "
          f"bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

    d = np.diff(b)
    # Count up by vmax in each bin
    counts, _ = np.histogram(data, bins=b, weights=1/clean['VMAX'])
    centers = 0.5 * (b[:-1] + b[1:])
    density = counts / d

    # Now to be able to handle very low densities on the edges, add 5 points that are 0.1 * the edge density and 0.1 away.
    # We just need the abundance matching to handle the first few rare objects without pulling values that are so extremal as to be not real.
    scale_length = 0.01 * (np.max(b) - np.min(b))  # 1% of the width of the features space we used
    tail =  density[0] * 10.0**np.arange(-1, -6, -1)
    centers_tail = centers[0] - scale_length * np.arange(1, 6)
    tail2 = density[-1] * 10.0**np.arange(-1, -6, -1)
    centers_tail2 = centers[-1] + scale_length * np.arange(1, 6)

    density = np.concatenate([tail[::-1], density, tail2])
    centers = np.concatenate([centers_tail[::-1], centers, centers_tail2])

    output_data = np.column_stack((centers, density))

    ax.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    ax2.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    np.savetxt(get_gal_ica_density_func_file("2p", i), output_data)

ax.legend()
ax2.set_xlim(-3.5, 3.5)
#ax2.set_ylim(3e-4, 0.03)
plt.tight_layout()
plt.show()

In [ ]:
spike_bin = clean.loc[(clean['ABS_MAG_R_K_BEST'] > -18.5) & (clean['ABS_MAG_R_K_BEST'] < -17.5)]
print(spike_bin[['ABS_MAG_R_K_BEST', 'Z', 'VMAX']].sort_values('VMAX'))

In [ ]:
###################################
# Volume Limited Density Functions
###################################
zmax = 0.1
frac_sky = y3_fullfoot_cat.GF_props['frac_area']
all_with_ica['ZMAX'] = get_max_observable_z(all_with_ica['ABSMAG01_SDSS_R'], 19.5)
vollim_sample = all_with_ica.loc[all_with_ica['ZMAX'] >= zmax]
vollim_sample = vollim_sample.loc[vollim_sample['Z'] <= zmax]
faintest = np.max(vollim_sample['ABSMAG01_SDSS_R'])
print(f"Volume limited sample size (z<={zmax}): {len(vollim_sample):,} / {len(all_with_ica):,}")
print(f"Faintest absolute magnitude in volume limited sample: {faintest:.2f}")

VOLUME = get_volume_at_z(zmax, frac_sky)
print(f"Volume of sample (z<={zmax}): {VOLUME:.2e} (Mpc/h)^3")

# Because we are only  using spec galaxies, we have fiber incompleteness. 
# We will correct for this by reducing the volume by the spectroscopic completeness.
# TODO IIP weights instead? Other ideas? 
# The absolute density should matter because it controls where we go on the halo density functions
total_targets = len(y3_fullfoot_cat.all_data)
total_spec = len(y3_fullfoot_cat.all_data.loc[y3_fullfoot_cat.all_data['Z_ASSIGNED_FLAG'] == AssignedRedshiftFlag.DESI_SPEC.value])
spec_completeness = total_spec / total_targets
VOLUME *= spec_completeness
print(f"Spectroscopic completeness: {spec_completeness:.2%}")
print(f"Effective volume after spectroscopic completeness correction: {VOLUME:.2e} (Mpc/h)^3")


# Normal Property Density Functions
fig, axes = plt.subplots(1, len(feature_cols), figsize=(5*len(feature_cols), 5))
for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.set_xlabel(f'{col}')
    ax.set_yscale('log')
    ax.set_ylabel('Density [1/(Mpc/h)^3]')
    ax.grid(True, ls="--", lw=0.5, alpha=0.5)

for col in feature_cols:
    data = vollim_sample[col].values

    b = make_adaptive_density_bins(data, 300, 0, 0.30, 1e-5)
    print(f"{col}: data [{data.min():.2f}, {data.max():.2f}]  bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

    d = np.diff(b)
    counts, _ = np.histogram(data, bins=b)
    centers = 0.5 * (b[:-1] + b[1:])
    density = counts / d / VOLUME

    # Now to be able to handle very low densities on the edges, add 5 points that are 0.1 * the edge density and 0.1 away.
    # We just need the abundance matching to handle the first few rare objects without pulling values that are so extremal as to be not real.
    scale_length = 0.01 * (np.max(b) - np.min(b))  # 1% of the width of the features space we used
    tail =  density[0] * 10.0**np.arange(-1, -6, -1)
    centers_tail = centers[0] - scale_length * np.arange(1, 6)
    tail2 = density[-1] * 10.0**np.arange(-1, -6, -1)
    centers_tail2 = centers[-1] + scale_length * np.arange(1, 6)

    density = np.concatenate([tail[::-1], density, tail2])
    centers = np.concatenate([centers_tail[::-1], centers, centers_tail2])

    # For magnitudes, reverse the innate order because we want more negative to be on the right (later in the file)
    if 'MAG' in col.upper():
        centers = centers[::-1]
        density = density[::-1]

    output_data = np.column_stack((centers, density))

    ax = axes[feature_cols.index(col)]
    ax.plot(centers, density, drawstyle='steps-mid', label=col, color=pp.get_color(feature_cols.index(col)))
    np.savetxt(get_gal_prop_density_func_file(col), output_data)

plt.tight_layout()
plt.show()

# ICA Density Functions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax2 = axes[1]
ax.set_xlabel('ICAx Value')
ax2.set_xlabel('ICAx Value (zoom)')
ax.set_yscale('log')
ax2.set_yscale('log')
ax.set_ylabel('Density [1/(Mpc/h)^3]')
ax2.set_ylabel('Density [1/(Mpc/h)^3]')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)
ax2.grid(True, ls="--", lw=0.5, alpha=0.5)

for i in range(all_ica_full.shape[1]):
    colname = f'ICA{i+1}'
    data = vollim_sample[colname].values

    b = make_adaptive_density_bins(data, 300, 20, 0.35, 1e-5)
    print(f"{colname}: data [{data.min():.2f}, {data.max():.2f}]  "
          f"bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

    dICA = np.diff(b)
    counts, _ = np.histogram(data, bins=b)
    centers = 0.5 * (b[:-1] + b[1:])
    density = counts / dICA / VOLUME

    # Now to be able to handle very low densities on the edges, add 5 points that are 0.1 * the edge density and 0.1 away.
    # We just need the abundance matching to handle the first few rare objects without pulling values that are so extremal as to be not real.
    scale_length = 0.01 * (np.max(b) - np.min(b))  # 1% of the width of the features space we used
    tail =  density[0] * 10.0**np.arange(-1, -6, -1)
    centers_tail = centers[0] - scale_length * np.arange(1, 6)
    tail2 = density[-1] * 10.0**np.arange(-1, -6, -1)
    centers_tail2 = centers[-1] + scale_length * np.arange(1, 6)

    density = np.concatenate([tail[::-1], density, tail2])
    centers = np.concatenate([centers_tail[::-1], centers, centers_tail2])

    output_data = np.column_stack((centers, density))

    ax.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    ax2.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    np.savetxt(get_gal_ica_density_func_file(i+1), output_data)

ax.legend()
ax2.set_xlim(-3.5, 3.5)
ax2.set_ylim(3e-4, 0.03)
plt.tight_layout()
plt.show()

# Show previous results

In [ ]:
features = ['ABS_MAG_R_K_BEST', 'G-R']

# Read in the density functions we just wrote out and plot them to verify they look correct
fig, axes = plt.subplots(1, len(features), figsize=(6*len(features), 5))
for i, col in enumerate(features):
    ax = axes[i]
    ax.set_xlabel(f'{col}')
    ax.set_yscale('log')
    ax.set_ylabel('Density [1/(Mpc/h)^3]')
    ax.grid(True, ls="--", lw=0.5, alpha=0.5)

    data = np.loadtxt(get_gal_prop_density_func_file(col))
    centers, density = data[:, 0], data[:, 1]
    ax.plot(centers, density, drawstyle='steps-mid', label=col, color=pp.get_color(i))
    ax.legend()